In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings 
warnings.filterwarnings('ignore')

In [2]:
crimes = pd.read_csv('datasets\\Chicago_Crimes.csv')

In [3]:
crimes.columns

Index(['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type',
       'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat',
       'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate',
       'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude',
       'Location'],
      dtype='object')

In [4]:
import folium
from folium.plugins import HeatMap

<h1>Crimes Heatmap</h1>
<hr>

In [5]:
crimes['Primary Type'].unique()

array(['THEFT', 'OTHER OFFENSE', 'MOTOR VEHICLE THEFT',
       'WEAPONS VIOLATION', 'BATTERY', 'ASSAULT',
       'CRIMINAL SEXUAL ASSAULT', 'CRIMINAL TRESPASS', 'CRIMINAL DAMAGE',
       'DECEPTIVE PRACTICE', 'SEX OFFENSE', 'ROBBERY', 'NARCOTICS',
       'HOMICIDE', 'INTERFERENCE WITH PUBLIC OFFICER', 'BURGLARY',
       'ARSON', 'OFFENSE INVOLVING CHILDREN', 'INTIMIDATION',
       'PUBLIC PEACE VIOLATION', 'CONCEALED CARRY LICENSE VIOLATION',
       'KIDNAPPING', 'STALKING', 'LIQUOR LAW VIOLATION', 'PROSTITUTION',
       'GAMBLING', 'OBSCENITY', 'PUBLIC INDECENCY', 'HUMAN TRAFFICKING',
       'OTHER NARCOTIC VIOLATION', 'NON-CRIMINAL'], dtype=object)

<h1> GeoSpatial Map of NARCOTICS </h1>

In [6]:
narcotics = crimes[crimes['Primary Type'] == "NARCOTICS"]

In [7]:
narcotics.columns

Index(['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type',
       'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat',
       'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate',
       'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude',
       'Location'],
      dtype='object')

In [8]:
narcotics_df = narcotics.groupby(['Longitude','Latitude']).size().reset_index(name = 'incident_count')

In [9]:
narcotics_df

,Longitude,Latitude,incident_count
0,-87.915105,41.953783,22
1,-87.906463,41.979006,5
2,-87.905227,41.976290,6
3,-87.900984,41.976763,1
4,-87.890372,41.974862,2
...,...,...,...
4685,-87.535974,41.702737,1
4686,-87.535953,41.722224,1
4687,-87.535282,41.707499,1
4688,-87.534604,41.711979,1


In [10]:
narcotics_df

# Normalize weights (OPTIONAL)
narcotics_df['normalized_weight'] = (narcotics_df['incident_count'] - narcotics_df['incident_count'].min()) / \
                                     (narcotics_df['incident_count'].max() - narcotics_df['incident_count'].min())

narcotics_list = narcotics_df[[ 'Latitude', 'Longitude','normalized_weight']].values.tolist()

# Create a base map
us = folium.Map(location=[ 41.8781, -87.6298], zoom_start=8)
# Latitude : 41.8781
# Longitude : -87.6298

# Add heatmap layer
HeatMap(narcotics_list).add_to(us)

# Save or display the map
us.save('narcotics.html')
us

In [11]:
narco_arrest = crimes[(crimes['Primary Type'] == "NARCOTICS") & (crimes['Arrest'] == True)]

In [12]:
narco_arrest_df = narcotics.groupby(['Longitude','Latitude']).size().reset_index(name = 'incident_count')

# Normalize weights (OPTIONAL)
narco_arrest_df['normalized_weight'] = (narco_arrest_df['incident_count'] - narco_arrest_df['incident_count'].min()) / \
                                     (narco_arrest_df['incident_count'].max() - narco_arrest_df['incident_count'].min())

narco_arrest_list = narcotics_df[[ 'Latitude', 'Longitude','normalized_weight']].values.tolist()

# Create a base map
us = folium.Map(location=[ 41.8781, -87.6298], zoom_start=10)
# Latitude : 41.8781
# Longitude : -87.6298

# Add heatmap layer
HeatMap(narco_arrest_list).add_to(us)

# Save or display the map
us.save('narcotics.html')
us

In [14]:
crime_df = crimes.dropna(subset=['Latitude', 'Longitude'])

In [15]:
import pandas as pd
import folium
from folium.plugins import HeatMap

crime_type = 'NARCOTICS'
crime_df = crimes[crimes['Primary Type'] == crime_type]
crime_df = crime_df.dropna(subset=['Latitude', 'Longitude'])

crime_grouped = crime_df.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')
crime_grouped['normalized_weight'] = (crime_grouped['incident_count'] - crime_grouped['incident_count'].min()) / (crime_grouped['incident_count'].max() - crime_grouped['incident_count'].min())

heatmap_data = crime_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298]
base_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(heatmap_data).add_to(base_map)
base_map.save(f'{crime_type.lower()}_heatmap.html')


<h1>Question 1: What is the correlation between THEFT incidents and the time of day they occur in Chicago?</h1>

In [16]:
theft = crimes[crimes['Primary Type'] == "THEFT"]

In [17]:
theft.columns

Index(['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type',
       'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat',
       'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate',
       'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude',
       'Location'],
      dtype='object')

In [57]:
theft = crimes[crimes['Primary Type'] == "THEFT"]
theft['hour'] = pd.to_datetime(theft['Date']).dt.hour

theft_by_hour = theft.groupby('hour').size().reset_index(name='incident_count')
theft_by_hour

,hour,incident_count
0,0,3021
1,1,1345
2,2,1216
3,3,1059
4,4,889
5,5,882
6,6,1057
7,7,1602
8,8,2107
9,9,2422


In [59]:
theft_by_location = theft.groupby(['Longitude', 'Latitude', 'hour']).size().reset_index(name='incident_count')

theft_by_location['normalized_weight'] = (theft_by_location['incident_count'] - theft_by_location['incident_count'].min()) / \
                                         (theft_by_location['incident_count'].max() - theft_by_location['incident_count'].min())

heatmap_data = theft_by_location[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()
map_center = [41.8781, -87.6298]  
base_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(heatmap_data).add_to(base_map)
base_map.save('theft_heatmap_by_hour.html')

base_map

<h1>Question 2: How does the distribution of MOTOR VEHICLE THEFT incidents differ between residential and commercial areas in Chicago?</h1>

In [66]:
motor_vehicle_theft = crimes[crimes['Primary Type'] == "MOTOR VEHICLE THEFT"]

motor_vehicle_theft_residential = motor_vehicle_theft[motor_vehicle_theft['Community Area'].isin([1, 2, 3, 4])]  
motor_vehicle_theft_commercial = motor_vehicle_theft[motor_vehicle_theft['Community Area'].isin([5, 6, 7, 8])]  

residential_df = motor_vehicle_theft_residential.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')
residential_df['normalized_weight'] = (residential_df['incident_count'] - residential_df['incident_count'].min()) / \
                                       (residential_df['incident_count'].max() - residential_df['incident_count'].min())
residential_list = residential_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

residential_list

[[42.000049573, -87.709367836, 0.0],
 [41.999882166, -87.709365793, 0.0],
 [41.997845884, -87.709348897, 0.0],
 [42.005384054, -87.709326751, 0.0],
 [41.996265073, -87.709320195, 0.0],
 [42.007142825, -87.709280128, 0.0],
 [41.99447279, -87.709260451, 0.0],
 [42.008981153, -87.709229045, 0.0],
 [42.009258281, -87.709222657, 0.2],
 [41.999225342, -87.708684249, 0.0],
 [41.997404798, -87.708473966, 0.0],
 [41.995639856, -87.708395584, 0.0],
 [41.991924515, -87.707946365, 0.0],
 [41.990686468, -87.707870201, 0.0],
 [41.999239502, -87.707757018, 0.0],
 [41.999241785, -87.707672381, 0.0],
 [42.015527306, -87.707446545, 0.0],
 [41.997425544, -87.707245033, 0.0],
 [41.996339599, -87.706880443, 0.0],
 [41.994703766, -87.70682659, 0.0],
 [42.014982916, -87.706752712, 0.0],
 [41.991209771, -87.706710038, 0.0],
 [41.987313756, -87.706343639, 0.0],
 [41.997444316, -87.706159593, 0.0],
 [41.995673845, -87.706073972, 0.0],
 [41.99301404, -87.705949267, 0.0],
 [41.988504845, -87.705861127, 0.0],
 [41

In [67]:
commercial_df = motor_vehicle_theft_commercial.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')
commercial_df['normalized_weight'] = (commercial_df['incident_count'] - commercial_df['incident_count'].min()) / \
                                      (commercial_df['incident_count'].max() - commercial_df['incident_count'].min())
commercial_list = commercial_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

commercial_list

[[41.954659535, -87.693245045, 0.0],
 [41.957878133, -87.691705449, 0.0],
 [41.956750335, -87.689757193, 0.0],
 [41.956268723, -87.6885341, 0.0],
 [41.954243156, -87.688473638, 0.0],
 [41.952610091, -87.688427603, 0.0],
 [41.945465738, -87.688220247, 0.0],
 [41.94492672, -87.687527277, 0.0],
 [41.945703199, -87.687015848, 0.0],
 [41.946755428, -87.686740554, 0.0],
 [41.957693443, -87.68666676, 0.0],
 [41.943123577, -87.686031082, 0.0],
 [41.950999024, -87.685951408, 0.0],
 [41.949511414, -87.685907643, 0.14285714285714285],
 [41.945885725, -87.685804645, 0.0],
 [41.945402667, -87.685791154, 0.0],
 [41.943580218, -87.685739756, 0.0],
 [41.942185945, -87.685702406, 0.0],
 [41.942131043, -87.685699286, 0.0],
 [41.942150252, -87.685699091, 0.0],
 [41.941760512, -87.685688333, 0.0],
 [41.940714797, -87.685658484, 0.0],
 [41.940742238, -87.685658206, 0.0],
 [41.937072722, -87.685570375, 0.0],
 [41.936424999, -87.685554878, 0.14285714285714285],
 [41.946776576, -87.685137641, 0.0],
 [41.96025

In [68]:
us_residential = folium.Map(location=[41.8781, -87.6298], zoom_start=8)
HeatMap(residential_list).add_to(us_residential)
us_residential.save('motor_vehicle_theft_residential_heatmap.html')
us_residential

In [69]:

us_commercial = folium.Map(location=[41.8781, -87.6298], zoom_start=8)
HeatMap(commercial_list).add_to(us_commercial)
us_commercial.save('motor_vehicle_theft_commercial_heatmap.html')
us_commercial

<h1>Question 3: Which WEAPONS VIOLATION incidents are concentrated near schools or educational institutions in Chicago?</h1>

In [72]:
weapons_violation = crimes[crimes['Primary Type'] == "WEAPONS VIOLATION"]

schools = pd.DataFrame({
    'name': ['School A', 'School B', 'School C'],  
    'latitude': [41.8800, 41.8900, 41.8750],  
    'longitude': [-87.6300, -87.6400, -87.6350]  
})

def is_near_school(incident_row, schools, radius=0.5):
    for _, school in schools.iterrows():
        distance = ((incident_row['Latitude'] - school['latitude'])**2 + (incident_row['Longitude'] - school['longitude'])**2)**0.5
        if distance < radius:  
            return True
    return False

near_school_rows = []

for index, row in weapons_violation.iterrows():
    if is_near_school(row, schools):
        near_school_rows.append(row)

weapons_near_school = pd.DataFrame(near_school_rows)

weapons_near_school_df = weapons_near_school.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

weapons_near_school_df['normalized_weight'] = (weapons_near_school_df['incident_count'] - weapons_near_school_df['incident_count'].min()) / \
                                               (weapons_near_school_df['incident_count'].max() - weapons_near_school_df['incident_count'].min())

weapons_near_school_df

,Longitude,Latitude,incident_count,normalized_weight
0,-87.906463,41.979006,6,0.416667
1,-87.904976,41.976421,2,0.083333
2,-87.900984,41.976763,2,0.083333
3,-87.890372,41.974862,4,0.250000
4,-87.844045,41.987481,1,0.000000
...,...,...,...,...
6589,-87.527258,41.651431,1,0.000000
6590,-87.527008,41.696426,1,0.000000
6591,-87.526975,41.700215,1,0.000000
6592,-87.526072,41.721219,1,0.000000


In [73]:
weapons_near_school_list = weapons_near_school_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

us_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
HeatMap(weapons_near_school_list).add_to(us_map)

us_map.save('weapons_violation_near_schools_heatmap.html')
us_map

<h1>Question 4: What is the distribution of BATTERY incidents in relation to public transportation stations (bus and train stations) in Chicago?</h1>

In [76]:
battery = crimes[crimes['Primary Type'] == "BATTERY"]

stations = pd.DataFrame({
    'name': ['Station A', 'Station B'],  
    'latitude': [41.8700, 41.8800], 
    'longitude': [-87.6300, -87.6400] 
})

def is_near_station(row, stations, radius=0.25):
    for _, station in stations.iterrows():
        station_location = (station['latitude'], station['longitude'])
        crime_location = (row['Latitude'], row['Longitude'])
        distance = ((crime_location[0] - station_location[0]) ** 2 + (crime_location[1] - station_location[1]) ** 2) ** 0.5  # Simple Euclidean distance
        if distance < radius:
            return True
    return False

battery['near_station'] = battery.apply(is_near_station, axis=1, stations=stations)

battery_near_station = battery[battery['near_station']]

battery_near_station_df = battery_near_station.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

battery_near_station_df

,Longitude,Latitude,incident_count
0,-87.846525,41.972467,2
1,-87.846517,41.972939,1
2,-87.846512,41.973189,2
3,-87.846497,41.974133,1
4,-87.846496,41.974311,3
...,...,...,...
29342,-87.525947,41.722695,1
29343,-87.525759,41.700229,1
29344,-87.525406,41.702685,1
29345,-87.525274,41.702684,2


In [77]:
battery_near_station_df['normalized_weight'] = (battery_near_station_df['incident_count'] - battery_near_station_df['incident_count'].min()) / \
                                               (battery_near_station_df['incident_count'].max() - battery_near_station_df['incident_count'].min())

battery_near_station_list = battery_near_station_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298] 
us_battery_near_station = folium.Map(location=map_center, zoom_start=10)

HeatMap(battery_near_station_list).add_to(us_battery_near_station)

us_battery_near_station.save('battery_near_station_heatmap.html')
us_battery_near_station

<h1> Question 5: How do ASSAULT incidents vary by day of the week in different neighborhoods of Chicago?</h1>

In [78]:
assault = crimes[crimes['Primary Type'] == "ASSAULT"]

assault['day_of_week'] = pd.to_datetime(assault['Date']).dt.day_name()

assault_by_day = assault.groupby(['day_of_week', 'Community Area']).size().reset_index(name='incident_count')

assault_day_of_week = assault_by_day[assault_by_day['day_of_week'] == "Monday"]

assault_day_of_week_df = assault[assault['day_of_week'] == 'Monday'] 
assault_day_of_week_grouped = assault_day_of_week_df.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

assault_day_of_week_grouped

,Longitude,Latitude,incident_count
0,-87.922965,41.961888,1
1,-87.900984,41.976763,1
2,-87.893869,41.985158,1
3,-87.890372,41.974862,2
4,-87.888400,41.975549,1
...,...,...,...
2966,-87.533562,41.711225,1
2967,-87.532869,41.717192,2
2968,-87.531658,41.704096,1
2969,-87.528126,41.704543,1


In [79]:
assault_day_of_week_grouped['normalized_weight'] = (assault_day_of_week_grouped['incident_count'] - assault_day_of_week_grouped['incident_count'].min()) / \
                                                   (assault_day_of_week_grouped['incident_count'].max() - assault_day_of_week_grouped['incident_count'].min())

assault_heatmap_data = assault_day_of_week_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298]  
assault_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(assault_heatmap_data).add_to(assault_map)

assault_map.save('assault_day_of_week_heatmap.html')

assault_map

<h1>Question 6: How many ASSAULT incidents occurred in each neighborhood of Chicago?</h1>

In [37]:
assault = crimes[crimes['Primary Type'] == "ASSAULT"]
assault_by_neighborhood = assault.groupby('Community Area')['ID'].count().reset_index(name='incident_count')
assault_by_neighborhood = assault_by_neighborhood.sort_values(by='incident_count', ascending=False)
assault_by_neighborhood

,Community Area,incident_count
24,25.0,1258
42,43.0,994
27,28.0,876
70,71.0,740
68,69.0,706
...,...,...
17,18.0,55
73,74.0,43
46,47.0,30
11,12.0,26


In [80]:
assault_with_neighborhood = pd.merge(assault, assault_by_neighborhood[['Community Area', 'incident_count']], on='Community Area')

assault_by_location = assault_with_neighborhood.groupby(['Longitude', 'Latitude', 'incident_count']).size().reset_index(name='count')

assault_by_location['normalized_weight'] = (assault_by_location['count'] - assault_by_location['count'].min()) / \
                                           (assault_by_location['count'].max() - assault_by_location['count'].min())

heatmap_data = assault_by_location[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298] 
assault_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(heatmap_data).add_to(assault_map)
assault_map.save('assault_by_neighborhood_heatmap.html')

assault_map

<h1>Question 7: What is the yearly trend of CRIMINAL SEXUAL ASSAULT incidents in Chicago?</h1>

In [38]:
criminal_sexual_assault = crimes[crimes['Primary Type'] == "CRIMINAL SEXUAL ASSAULT"]
criminal_sexual_assault['Year'] = pd.to_datetime(criminal_sexual_assault['Date']).dt.year
assault_yearly_trend = criminal_sexual_assault.groupby('Year')['ID'].count().reset_index(name='incident_count')
assault_yearly_trend

,Year,incident_count
0,2024,1155
1,2025,418


In [81]:
criminal_sexual_assault_grouped = criminal_sexual_assault.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

criminal_sexual_assault_grouped['normalized_weight'] = (criminal_sexual_assault_grouped['incident_count'] - criminal_sexual_assault_grouped['incident_count'].min()) / \
                                                      (criminal_sexual_assault_grouped['incident_count'].max() - criminal_sexual_assault_grouped['incident_count'].min())

criminal_sexual_assault_heatmap_data = criminal_sexual_assault_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298]  
assault_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(criminal_sexual_assault_heatmap_data).add_to(assault_map)
assault_map.save('criminal_sexual_assault_heatmap.html')

assault_map

<h1>Question 8: What is the ratio of arrests in CRIMINAL TRESPASS cases in Chicago?</h1>

In [39]:
criminal_trespass = crimes[crimes['Primary Type'] == "CRIMINAL TRESPASS"]
arrest_ratio_trespass = criminal_trespass.groupby('Arrest')['ID'].count().reset_index(name='incident_count')
arrest_ratio_trespass['ratio'] = arrest_ratio_trespass['incident_count'] / arrest_ratio_trespass['incident_count'].sum()
arrest_ratio_trespass

,Arrest,incident_count,ratio
0,False,3569,0.707433
1,True,1476,0.292567


In [82]:
criminal_trespass_grouped = criminal_trespass.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

criminal_trespass_grouped['normalized_weight'] = (criminal_trespass_grouped['incident_count'] - criminal_trespass_grouped['incident_count'].min()) / \
                                                 (criminal_trespass_grouped['incident_count'].max() - criminal_trespass_grouped['incident_count'].min())

criminal_trespass_heatmap_data = criminal_trespass_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298] 
trespass_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(criminal_trespass_heatmap_data).add_to(trespass_map)

trespass_map.save('criminal_trespass_heatmap.html')
trespass_map

<h1>Question 9: What day of the week has the highest number of CRIMINAL DAMAGE incidents?</h1>

In [40]:
criminal_damage = crimes[crimes['Primary Type'] == "CRIMINAL DAMAGE"]
criminal_damage['DayOfWeek'] = pd.to_datetime(criminal_damage['Date']).dt.day_name()
damage_by_day = criminal_damage.groupby('DayOfWeek')['ID'].count().reset_index(name='incident_count')
damage_by_day = damage_by_day.sort_values(by='incident_count', ascending=False)
damage_by_day


,DayOfWeek,incident_count
3,Sunday,4285
2,Saturday,4221
1,Monday,3935
0,Friday,3817
5,Tuesday,3757
6,Wednesday,3567
4,Thursday,3512


In [83]:
criminal_damage_df = criminal_damage.dropna(subset=['Longitude', 'Latitude'])  # Drop rows without lat/long
damage_grouped = criminal_damage_df.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

damage_grouped['normalized_weight'] = (damage_grouped['incident_count'] - damage_grouped['incident_count'].min()) / \
                                      (damage_grouped['incident_count'].max() - damage_grouped['incident_count'].min())

damage_heatmap_data = damage_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298]  
damage_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(damage_heatmap_data).add_to(damage_map)

damage_map.save('criminal_damage_heatmap.html')

damage_map

<h1>Question 10: How do DECEPTIVE PRACTICE incidents vary by ward in Chicago?</h1>

In [41]:
deceptive_practice = crimes[crimes['Primary Type'] == "DECEPTIVE PRACTICE"]
deceptive_practice_by_ward = deceptive_practice.groupby('Ward')['ID'].count().reset_index(name='incident_count')
deceptive_practice_by_ward = deceptive_practice_by_ward.sort_values(by='incident_count', ascending=False)
deceptive_practice_by_ward


,Ward,incident_count
41,42,1023
33,34,629
26,27,571
3,4,506
40,41,419
1,2,414
20,21,414
43,44,408
27,28,396
5,6,375


In [84]:
deceptive_practice_df = deceptive_practice.dropna(subset=['Latitude', 'Longitude'])

deceptive_practice_grouped = deceptive_practice_df.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

deceptive_practice_grouped['normalized_weight'] = (deceptive_practice_grouped['incident_count'] - deceptive_practice_grouped['incident_count'].min()) / \
                                                 (deceptive_practice_grouped['incident_count'].max() - deceptive_practice_grouped['incident_count'].min())

deceptive_practice_heatmap_data = deceptive_practice_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

map_center = [41.8781, -87.6298]  
deceptive_practice_map = folium.Map(location=map_center, zoom_start=10)

HeatMap(deceptive_practice_heatmap_data).add_to(deceptive_practice_map)
deceptive_practice_map.save('deceptive_practice_heatmap.html')

deceptive_practice_map

<h1>Question 11: What is the temporal pattern of SEX OFFENSE incidents in Chicago throughout the day?</h1>

In [85]:
sex_offense = crimes[crimes['Primary Type'] == "SEX OFFENSE"]
sex_offense['hour'] = pd.to_datetime(sex_offense['Date']).dt.hour

sex_offense_by_hour = sex_offense.groupby('hour').size().reset_index(name='incident_count')
sex_offense_by_hour

,hour,incident_count
0,0,168
1,1,29
2,2,31
3,3,21
4,4,15
5,5,25
6,6,19
7,7,42
8,8,58
9,9,55


In [97]:
sex_offense_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

sex_offense_grouped = sex_offense.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

sex_offense_grouped['normalized_weight'] = (sex_offense_grouped['incident_count'] - sex_offense_grouped['incident_count'].min()) / \
                                           (sex_offense_grouped['incident_count'].max() - sex_offense_grouped['incident_count'].min())

sex_offense_list = sex_offense_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

HeatMap(sex_offense_list).add_to(sex_offense_map)

sex_offense_map.save('sex_offense_heatmap_by_hour.html')

sex_offense_map

<h1>Question 12: Which areas have the highest concentration of ROBBERY incidents in Chicago?</h1>

In [86]:
robbery = crimes[crimes['Primary Type'] == "ROBBERY"]
robbery_df = robbery.dropna(subset=['Latitude', 'Longitude'])

robbery_grouped = robbery_df.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')
robbery_grouped['normalized_weight'] = (robbery_grouped['incident_count'] - robbery_grouped['incident_count'].min()) / \
                                       (robbery_grouped['incident_count'].max() - robbery_grouped['incident_count'].min())

robbery_grouped

,Longitude,Latitude,incident_count,normalized_weight
0,-87.906463,41.979006,1,0.000000
1,-87.876421,41.976182,1,0.000000
2,-87.843179,41.973756,1,0.000000
3,-87.841540,41.976546,1,0.000000
4,-87.819962,41.952470,1,0.000000
...,...,...,...,...
6814,-87.532869,41.705929,1,0.000000
6815,-87.527856,41.702697,1,0.000000
6816,-87.527258,41.651431,1,0.000000
6817,-87.526058,41.702687,1,0.000000


In [87]:
robbery_heatmap_data = robbery_grouped[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

robbery_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)
HeatMap(robbery_heatmap_data).add_to(robbery_map)

robbery_map.save('robbery_heatmap.html')
robbery_map

<h1>Question 13: What is the spatial distribution of HOMICIDE cases across Chicago?</h1>

In [47]:
homicide = crimes[crimes['Primary Type'] == "HOMICIDE"]
homicide_df = homicide.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

homicide_df

,Longitude,Latitude,incident_count
0,-87.832993,41.951138,1
1,-87.817771,41.991139,1
2,-87.797136,41.792046,1
3,-87.796229,41.916768,1
4,-87.788530,41.919542,1
...,...,...,...
515,-87.545769,41.734625,1
516,-87.538921,41.722409,1
517,-87.537455,41.652669,1
518,-87.534823,41.737771,2


In [48]:
homicide_df['normalized_weight'] = (homicide_df['incident_count'] - homicide_df['incident_count'].min()) / \
                                    (homicide_df['incident_count'].max() - homicide_df['incident_count'].min())

homicide_list = homicide_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

us = folium.Map(location=[41.8781, -87.6298], zoom_start=8)
HeatMap(homicide_list).add_to(us)

us.save('homicide_heatmap.html')
us

<h1>Question 14: How do BURGLARY incidents vary by community area in Chicago?</h1>

In [99]:
burglary = crimes[crimes['Primary Type'] == "BURGLARY"]
burglary_by_area = burglary.groupby('Community Area')['ID'].count().reset_index(name='incident_count')
burglary_by_area = burglary_by_area.sort_values(by='incident_count', ascending=False)

burglary_by_area

,Community Area,incident_count
23,24.0,466
42,43.0,409
27,28.0,357
24,25.0,334
5,6.0,290
...,...,...
35,36.0,16
75,76.0,15
17,18.0,14
46,47.0,9


In [101]:
community_areas_coords = pd.DataFrame({
    'Community Area': range(1, 78),
    'Latitude': [41.8781 + 0.01 * i for i in range(77)],
    'Longitude': [-87.6298 - 0.01 * i for i in range(77)]
})

burglary_with_coords = burglary_by_area.merge(community_areas_coords, on='Community Area', how='left')

burglary_with_coords = burglary_with_coords.dropna(subset=['Latitude', 'Longitude'])

burglary_with_coords['normalized_weight'] = (burglary_with_coords['incident_count'] - burglary_with_coords['incident_count'].min()) / \
                                            (burglary_with_coords['incident_count'].max() - burglary_with_coords['incident_count'].min())

burglary_heatmap_data = burglary_with_coords[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

burglary_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

HeatMap(burglary_heatmap_data).add_to(burglary_map)

burglary_map.save('burglary_heatmap.html')

burglary_map


<h1>Question 15: What is the relationship between ARSON incidents and proximity to major roads or highways in Chicago?</h1>

In [95]:
arson = crimes[crimes['Primary Type'] == "ARSON"]

roads = pd.DataFrame({
    'name': ['Road A', 'Road B'], 
    'latitude': [41.8500, 41.8600],
    'longitude': [-87.6200, -87.6300]
})

def is_near_road(row, roads, distance_threshold=0.5):
    for _, road in roads.iterrows():
        lat_diff = row['Latitude'] - road['latitude']
        lon_diff = row['Longitude'] - road['longitude']
        distance = (lat_diff ** 2 + lon_diff ** 2) ** 0.5
        if distance < distance_threshold:
            return True
    return False

near_road_arson = []

for index, row in arson.iterrows():
    if is_near_road(row, roads):
        near_road_arson.append(row)

arson_near_road = pd.DataFrame(near_road_arson)

arson_near_road_df = arson_near_road.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')
arson_near_road_df

,Longitude,Latitude,incident_count
0,-87.810405,41.938749,1
1,-87.805588,41.940344,1
2,-87.795699,41.792077,1
3,-87.793274,41.938711,1
4,-87.792922,41.927697,1
...,...,...,...
435,-87.538910,41.698630,1
436,-87.534078,41.707425,1
437,-87.532873,41.707685,1
438,-87.532866,41.698541,1


In [96]:
arson_near_road_df['normalized_weight'] = (arson_near_road_df['incident_count'] - arson_near_road_df['incident_count'].min()) / \
                                           (arson_near_road_df['incident_count'].max() - arson_near_road_df['incident_count'].min())

arson_near_road_list = arson_near_road_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

arson_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

HeatMap(arson_near_road_list).add_to(arson_map)

arson_map.save('arson_near_road_heatmap.html')

arson_map

<h1>Question 16: What is the distribution of PROSTITUTION incidents across different neighborhoods in Chicago?</h1>

In [107]:
prostitution = crimes[crimes['Primary Type'] == "PROSTITUTION"]

prostitution_by_area = prostitution.groupby('Community Area')['ID'].count().reset_index(name='incident_count')
prostitution_by_area


,Community Area,incident_count
0,6.0,1
1,8.0,3
2,10.0,1
3,11.0,1
4,13.0,1
5,14.0,2
6,16.0,2
7,19.0,3
8,21.0,1
9,23.0,9


In [108]:
community_area_coordinates = {
    1: (41.8781, -87.6298),
    2: (41.8821, -87.6300)
}

prostitution_with_coords = prostitution_by_area.copy()

prostitution_with_coords['Latitude'] = [community_area_coordinates.get(area, (None, None))[0] 
                                         for area in prostitution_with_coords['Community Area']]
prostitution_with_coords['Longitude'] = [community_area_coordinates.get(area, (None, None))[1] 
                                          for area in prostitution_with_coords['Community Area']]

prostitution_with_coords.dropna(subset=['Latitude', 'Longitude'], inplace=True)

heatmap_data = prostitution_with_coords[['Latitude', 'Longitude', 'incident_count']].values.tolist()

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

HeatMap(heatmap_data).add_to(chicago_map)

chicago_map.save('prostitution_by_area_heatmap.html')

chicago_map

<h1>Question 17: How do VANDALISM incidents distribute across different times of the day in Chicago?</h1>

In [121]:
vandalism = crimes[crimes['Primary Type'] == "VANDALISM"]

vandalism['hour'] = pd.to_datetime(vandalism['Date']).dt.hour

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

vandalism['time_of_day'] = vandalism['hour'].apply(get_time_of_day)

vandalism_by_time_of_day = vandalism.groupby('time_of_day').size().reset_index(name='incident_count')
vandalism_by_time_of_day

,time_of_day,incident_count


In [122]:
time_of_day_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

for index, row in vandalism_by_time_of_day.iterrows():
    time_of_day_lat = 41.8781 + index * 0.01
    time_of_day_lon = -87.6298 - index * 0.01
    folium.Marker(
        location=[time_of_day_lat, time_of_day_lon],
        popup=f"Time of Day: {row['time_of_day']}, Incidents: {row['incident_count']}"
    ).add_to(time_of_day_map)

time_of_day_map.save('vandalism_time_of_day_map.html')

time_of_day_map

<h1>Question 18: What is the relationship between FRAUD incidents and proximity to commercial areas in Chicago?</h1>

In [124]:
fraud = crimes[crimes['Primary Type'] == "FRAUD"]

commercial_areas = pd.DataFrame({
    'name': ['Commercial Area A', 'Commercial Area B', 'Commercial Area C'],
    'latitude': [41.8789, 41.8900, 41.8750],
    'longitude': [-87.6298, -87.6350, -87.6225]
})

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    radius = 3958.8
    return radius * c

def distance_to_commercial_area(row, commercial_areas):
    crime_location = (row['Latitude'], row['Longitude'])
    min_distance = float('inf')
    
    for _, area in commercial_areas.iterrows():
        area_location = (area['latitude'], area['longitude'])
        distance = haversine(crime_location[0], crime_location[1], area_location[0], area_location[1])
        if distance < min_distance:
            min_distance = distance
            
    return min_distance

fraud['distance_to_commercial_area'] = fraud.apply(distance_to_commercial_area, axis=1, commercial_areas=commercial_areas)

proximity_threshold = 1

fraud_near_commercial = fraud[fraud['distance_to_commercial_area'] <= proximity_threshold]

fraud_near_commercial_df = fraud_near_commercial.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

fraud_near_commercial_df['normalized_weight'] = (fraud_near_commercial_df['incident_count'] - fraud_near_commercial_df['incident_count'].min()) / \
                                                 (fraud_near_commercial_df['incident_count'].max() - fraud_near_commercial_df['incident_count'].min())

heatmap_data = fraud_near_commercial_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

HeatMap(heatmap_data).add_to(chicago_map)

chicago_map.save('fraud_near_commercial_area_heatmap.html')

chicago_map


<h1>Question 19: What is the trend of BATTERY incidents over different months of the year in Chicago?</h1>

In [125]:
battery = crimes[crimes['Primary Type'] == "BATTERY"]

battery['month'] = pd.to_datetime(battery['Date']).dt.month

battery_by_month = battery.groupby('month').size().reset_index(name='incident_count')

battery_by_location = battery.groupby(['Latitude', 'Longitude']).size().reset_index(name='incident_count')

battery_by_location['normalized_weight'] = (battery_by_location['incident_count'] - battery_by_location['incident_count'].min()) / \
                                            (battery_by_location['incident_count'].max() - battery_by_location['incident_count'].min())

heatmap_data = battery_by_location[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

HeatMap(heatmap_data).add_to(chicago_map)

chicago_map.save('battery_incidents_heatmap.html')

chicago_map


<h1>Question 20: What is the distribution of ASSAULT incidents near hospitals in Chicago?</h1>

In [126]:
assault = crimes[crimes['Primary Type'] == "ASSAULT"]

hospitals = pd.DataFrame({
    'name': ['Hospital A', 'Hospital B', 'Hospital C'],
    'latitude': [41.8781, 41.8900, 41.8800],
    'longitude': [-87.6298, -87.6350, -87.6225]
})

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    radius = 3958.8
    return radius * c

def distance_to_hospital(row, hospitals):
    crime_location = (row['Latitude'], row['Longitude'])
    min_distance = float('inf')
    
    for _, hospital in hospitals.iterrows():
        hospital_location = (hospital['latitude'], hospital['longitude'])
        distance = haversine(crime_location[0], crime_location[1], hospital_location[0], hospital_location[1])
        if distance < min_distance:
            min_distance = distance
            
    return min_distance

assault['distance_to_hospital'] = assault.apply(distance_to_hospital, axis=1, hospitals=hospitals)

proximity_threshold = 1

assault_near_hospital = assault[assault['distance_to_hospital'] <= proximity_threshold]

assault_near_hospital_df = assault_near_hospital.groupby(['Longitude', 'Latitude']).size().reset_index(name='incident_count')

assault_near_hospital_df['normalized_weight'] = (assault_near_hospital_df['incident_count'] - assault_near_hospital_df['incident_count'].min()) / \
                                                 (assault_near_hospital_df['incident_count'].max() - assault_near_hospital_df['incident_count'].min())

heatmap_data = assault_near_hospital_df[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=12)

HeatMap(heatmap_data).add_to(chicago_map)

chicago_map.save('assault_near_hospital_heatmap.html')

chicago_map


<h1>Question 21: How are incidents of SHOPLIFTING distributed across neighborhoods in Chicago?</h1>

In [127]:
shoplifting = crimes[crimes['Primary Type'] == "SHOPLIFTING"]

shoplifting_by_area = shoplifting.groupby('Community Area')['ID'].count().reset_index(name='incident_count')
shoplifting_by_area = shoplifting_by_area.sort_values(by='incident_count', ascending=False)

community_area_coords = pd.DataFrame({
    'Community Area': [1, 2, 3, 4, 5],
    'Latitude': [41.8781, 41.8850, 41.8931, 41.9000, 41.9100],
    'Longitude': [-87.6298, -87.6350, -87.6400, -87.6500, -87.6550]
})

shoplifting_with_coords = shoplifting_by_area.merge(community_area_coords, on='Community Area', how='left')

chicago_map = folium.Map(location=[41.8781, -87.6298], zoom_start=11)

heatmap_data = shoplifting_with_coords[['Latitude', 'Longitude', 'incident_count']].values.tolist()

shoplifting_with_coords['normalized_weight'] = (shoplifting_with_coords['incident_count'] - shoplifting_with_coords['incident_count'].min()) / \
                                               (shoplifting_with_coords['incident_count'].max() - shoplifting_with_coords['incident_count'].min())

heatmap_data_normalized = shoplifting_with_coords[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()

HeatMap(heatmap_data_normalized).add_to(chicago_map)

chicago_map.save('shoplifting_heatmap.html')

chicago_map


<h1>Question 22: What is the trend of GAMBLING incidents across different times of the day in Chicago?</h1>

In [112]:
gambling = crimes[crimes['Primary Type'] == "GAMBLING"]

gambling['hour'] = pd.to_datetime(gambling['Date']).dt.hour

gambling_by_hour = gambling.groupby('hour').size().reset_index(name='incident_count')
gambling_by_hour

,hour,incident_count
0,12,3
1,13,1
2,14,3
3,15,1
4,16,1
5,17,1
6,18,1
7,20,2
8,21,2
9,22,1


In [113]:
gambling_by_hour_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

gambling_heatmap_data = gambling[['Latitude', 'Longitude', 'hour']].dropna()

HeatMap(gambling_heatmap_data[['Latitude', 'Longitude', 'hour']].values.tolist()).add_to(gambling_by_hour_map)

gambling_by_hour_map.save('gambling_by_hour_heatmap.html')

gambling_by_hour_map

<h1>Question 23: Where are the most concentrated PUBLIC INDECENCY incidents in Chicago?</h1>

In [129]:
public_indecency = crimes[crimes['Primary Type'] == "PUBLIC INDECENCY"]

public_indecency_by_location = public_indecency.groupby(['Latitude', 'Longitude']).size().reset_index(name='incident_count')
public_indecency_by_location

,Latitude,Longitude,incident_count
0,41.695908,-87.595823,1
1,41.765785,-87.683271,1
2,41.768692,-87.624942,1
3,41.784515,-87.653412,1
4,41.796151,-87.759560,1
5,41.852024,-87.686627,1
6,41.852260,-87.670225,1
7,41.878170,-87.629107,1
8,41.962324,-87.684443,1
9,41.990128,-87.670852,1


In [131]:
public_indecency = crimes[crimes['Primary Type'] == "PUBLIC INDECENCY"]

public_indecency_by_location = public_indecency.groupby(['Latitude', 'Longitude']).size().reset_index(name='incident_count')

public_indecency_by_location['normalized_weight'] = (public_indecency_by_location['incident_count'] - public_indecency_by_location['incident_count'].min()) / \
                                                   (public_indecency_by_location['incident_count'].max() - public_indecency_by_location['incident_count'].min())

public_indecency_by_location = public_indecency_by_location.dropna(subset=['Latitude', 'Longitude', 'normalized_weight'])
public_indecency_map = folium.Map(location=[41.8781, -87.6298], zoom_start=10)

public_indecency_heatmap_data = public_indecency_by_location[['Latitude', 'Longitude', 'normalized_weight']].values.tolist()
HeatMap(public_indecency_heatmap_data).add_to(public_indecency_map)

public_indecency_map.save('public_indecency_heatmap.html')

public_indecency_map
